# **LABELING DATA AVETA**




Library

In [ ]:
! pip install transformers torch pandas openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from transformers import pipeline

In [ ]:
# Membaca Data
# Pastikan kolom yang akan dilabeli bernama 'tokening' (hasil preprocessing)
data = pd.read_csv("data_aveta.csv")  # ubah sesuai nama file kamu
data = data.loc[:, ~data.columns.str.contains("^Unnamed")]
print(f"Total data: {len(data)}")
data.head()

Total data: 2449


,rating,ulasan,labeling manual
0,5/5,Hotel yang sgt strategis di jalan malioboro. K...,Positif
1,5/5,Lokasi yang top. Di sekitar lokasi hotel banya...,Positif
2,2/5,Kurang saya rekomendasikan. Pesan premier room...,Negatif
3,5/5,nginap 2 hari dan 2 malam bersama istri dan an...,Positif
4,2/5,Saya memberikan bintang 2 ini sebelum saya mas...,Negatif


**Transformer Lbeling**

In [ ]:
#Inisialisasi Model Transformer
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    truncation=True,
    max_length=512
)


Device set to use cpu


In [ ]:
# Jalankan analisis untuk setiap teks
def get_label_result(text):
    if pd.isna(text):
        return np.nan
    try:
        result = sentiment_analyzer(
            text,
            truncation=True,
            max_length=512
        )[0]
        label = result["label"].upper()
        if label == "POSITIVE":
            return 1
        elif label == "NEGATIVE":
            return 0
        else:
            return np.nan
    except Exception as e:
        return np.nan
# Terapkan ke kolom 'Ulasan'
data['label'] = data['ulasan'].apply(get_label_result)

In [ ]:
# Hitung distribusi hasil labeling
hasil_label = data['label'].value_counts(dropna=False)
print("Statistik Labeling")
print(f"Total data       : {len(data)}")
print(f"Positif (1)      : {hasil_label.get(1, 0)}")
print(f"Negatif (0)      : {hasil_label.get(0, 0)}")
print(f"Belum terlabeli  : {hasil_label.get(np.nan, 0)}")
print("\n Persentase")
print((hasil_label / len(data) * 100).round(2))

Statistik Labeling
Total data       : 2449
Positif (1)      : 2242
Negatif (0)      : 188
Belum terlabeli  : 19

 Persentase
label
1.0    91.55
0.0     7.68
NaN     0.78
Name: count, dtype: float64


In [ ]:
def label(label):
    if label == 1:
        return "Positif"
    elif label == 0:
        return "Negatif"
    else:
        return None
data["label"] = data["label"].apply(label)

In [ ]:
data

,rating,ulasan,labeling manual,label
0,5/5,Hotel yang sgt strategis di jalan malioboro. K...,Positif,Negatif
1,5/5,Lokasi yang top. Di sekitar lokasi hotel banya...,Positif,Positif
2,2/5,Kurang saya rekomendasikan. Pesan premier room...,Negatif,Negatif
3,5/5,nginap 2 hari dan 2 malam bersama istri dan an...,Positif,Positif
4,2/5,Saya memberikan bintang 2 ini sebelum saya mas...,Negatif,Negatif
...,...,...,...,...
2444,5/5,Semoga beruntung 👍🏻,Negatif,Positif
2445,4/5,Playanan 👍👍👍,Positif,Positif
2446,1/5,Manyalah..,Negatif,Negatif
2447,5/5,Daerah Malioboro,Positif,Positif


**Negatif**

In [ ]:
df_negatif = data[data['label'] == "Negatif"]
print(df_negatif)

     rating                                             ulasan  \
0       5/5  Hotel yang sgt strategis di jalan malioboro. K...   
2       2/5  Kurang saya rekomendasikan. Pesan premier room...   
4       2/5  Saya memberikan bintang 2 ini sebelum saya mas...   
6       1/5  Pengalaman menginap yang kurang menyenangkan  ...   
22      1/5  Mati listrik sampai 20 menit, tidak ada penjel...   
...     ...                                                ...   
2400    3/5  Hotel yang terletak di jantung Malioboro ini m...   
2430    5/5                    tidak bakalan balik lagiiiiii..   
2435    5/5                   Pengalaman yang luar biasa jelek   
2439    4/5                                Nongkring ajalah...   
2446    1/5                                         Manyalah..   

     labeling manual    label  
0            Positif  Negatif  
2            Negatif  Negatif  
4            Negatif  Negatif  
6            Negatif  Negatif  
22           Negatif  Negatif  
...            

In [ ]:
for idx, ulasan in df_negatif['ulasan'].items():
    print(f"{idx}: {ulasan}")
    print('-' * 50)#negatif

0: Hotel yang sgt strategis di jalan malioboro. Karyawan ramah2. Kamar bersih. House keeping jg baik, bersih dan cepat kerjanya.Minusnya: 1. Tidak ada lahan parkir. Sangat terbatas harus valet semua ninggal kunci. Bikin tdk nyaman.2. Kamar mandi tdk ada hand shower kalau bawa baby akan kesusahan utk mandi krn hanya shower yg ada di atap saja 3. Untuk breakfastnya variasinya lumayan hanya saja rasanya sangat kurang. Mnrt sy tdk enak. Hehehe. Harusnya bisa lbh enak lg. Semoga ditingkatkan 4. Kamar tidak kedap. Terutama kalau tarik kursi sangat jelas dr kamar sebelah Semoga ada perbaikan yah. Hotel ini sangat rekomended untuk dicoba. Semoga pelayanan dan kebersihannya dijaga sama bahkan lbh baik lg.
--------------------------------------------------
2: Kurang saya rekomendasikan. Pesan premier room di link nya aveta hotel, gak dapet bf. Padahal waktu check inn, dikabari kita free bf. Setelah esok hari mo bf, nomor kamar kita gak ada dlm list bf alias exclude bf!
Kamar cukup bersih dan sem

Unlabeled

In [ ]:
df_unlabeled = data[data['label'].isna()]
print(df_unlabeled)

     rating                                             ulasan  \
90      4/5  Studio anak harusnya ada jadwal sore dan malam...   
98      5/5  Saya booking 2 kamar untuk 4 malam, tapi saat ...   
133     5/5  Rombongan Tamu pakai Hiace Premio Luxury 8 sea...   
1136    1/5  maaf pak kmrn pak GM blg saya disuruh email da...   
1258    3/5  Akses kendaraan masuk hotel, menjadi kendala k...   
1426    3/5  Masih banyak perbaikan yang perlu di perbaiki,...   
1499    5/5  Firdaus , isa , aulia , yuni terima kasih untu...   
1553    5/5  Depan jalan malioboro, namun untuk skrng krna ...   
1617    5/5                        Mimpi itu menjadi kenyataan   
1646    3/5  Parkirnya perlu desain yg bagus fan dr depan t...   
1773    5/5  Sdh membuka akses jalan kecil samping maliobor...   
1780    5/5                      Great, selangkah ke malioboro   
1860    1/5           hotel masih baru dan dipaksakan sdh buka   
1935    5/5   Turun kamar hotel langsung Malioboro\nMuaantabbb   
2066    4/

In [ ]:
for idx, ulasan in df_unlabeled['ulasan'].items():
    print(f"{idx}: {ulasan}")
    print('-' * 50) #unlabeled

90: Studio anak harusnya ada jadwal sore dan malam dan anak usia 7 thn biasanya free tp ini bayar.
--------------------------------------------------
98: Saya booking 2 kamar untuk 4 malam, tapi saat mau check in disuruh nunggu jam 15:00 WIB, mohon penjelasan krn biasanya jam 14:00 sudah boleh CI
--------------------------------------------------
133: Rombongan Tamu pakai Hiace Premio Luxury 8 seat menginap di Aveta Hotel Malioboro
--------------------------------------------------
1136: maaf pak kmrn pak GM blg saya disuruh email dan beri no whatsapp sudah sy info dan sdh sy email tetapi sy tdk mendapatkan balasan smp dgn hari ini ya. dan utk booking, mmg tdk menggunakan nama sy. menggunakan nama “adi rachman” tgl 4-6 oct boleh silahkan d check. terimakasih.
--------------------------------------------------
1258: Akses kendaraan masuk hotel, menjadi kendala kelancaran manakala terjadi penutupan jalan Malioboro dari jam 18-21...
--------------------------------------------------
1426:

In [ ]:
# data = data.drop(columns=["label tranformer"])

In [ ]:
# # Simpan Hasil ke File
# data.to_csv("labeling_transformer_aveta.csv")